In [ ]:
from google.colab import files; files.upload()

Saving delhi_test.csv to delhi_test.csv
Saving delhi_train.csv to delhi_train.csv


In [ ]:
from google.colab import files; files.upload()


Saving scaler.pkl to scaler.pkl


{'scaler.pkl': b'\x80\x04\x95\xe8\n\x00\x00\x00\x00\x00\x00\x8c\x1bsklearn.preprocessing._data\x94\x8c\x0cMinMaxScaler\x94\x93\x94)\x81\x94}\x94(\x8c\rfeature_range\x94K\x00K\x01\x86\x94\x8c\x04copy\x94\x88\x8c\x04clip\x94\x89\x8c\x11feature_names_in_\x94\x8c\x16numpy._core.multiarray\x94\x8c\x0c_reconstruct\x94\x93\x94\x8c\x05numpy\x94\x8c\x07ndarray\x94\x93\x94K\x00\x85\x94C\x01b\x94\x87\x94R\x94(K\x01K*\x85\x94h\r\x8c\x05dtype\x94\x93\x94\x8c\x02O8\x94\x89\x88\x87\x94R\x94(K\x03\x8c\x01|\x94NNNJ\xff\xff\xff\xffJ\xff\xff\xff\xffK?t\x94b\x89]\x94(\x8c\x05PM2.5\x94\x8c\x04PM10\x94\x8c\x02NO\x94\x8c\x03NO2\x94\x8c\x03NOx\x94\x8c\x03NH3\x94\x8c\x02CO\x94\x8c\x03SO2\x94\x8c\x02O3\x94\x8c\x07Benzene\x94\x8c\x07Toluene\x94\x8c\x03AQI\x94\x8c\x08Latitude\x94\x8c\tLongitude\x94\x8c\rTemperature_C\x94\x8c\x0cHumidity_pct\x94\x8c\rWindSpeed_kmh\x94\x8c\x0cPressure_hPa\x94\x8c\x04hour\x94\x8c\x0bday_of_week\x94\x8c\x05month\x94\x8c\nis_weekend\x94\x8c\x0cPM2.5_roll3h\x94\x8c\rPM2.5_roll24h\x94\x

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
print(os.listdir('/content/drive/MyDrive'))

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
['Colab Notebooks', 'Vaishnavi_ML', 'Untitled presentation.gslides', 'Untitled document (5).gdoc', 'sonic-vs.zip', 'CV.gdoc', 'Untitled document (4).gdoc', 'IMG-20251219-WA0003.jpg', 'conference-template-a4.docx', 'Untitled site.gsite', 'Untitled spreadsheet.gsheet', 'arcade.zip', 'arcade', 'CEVI-R3.gslides', 'Screen Recording 2026-01-07 105713.mp4', '7_B.gslides', 'Light Green Illustrative Modern Group Project Presentation (1).gslides', 'Light Green Illustrative Modern Group Project Presentation.gslides', 'Untitled document (3).gdoc', "I am sharing 'I3CTCON 2026 PRESENTATION TEMPLETE (1) [Autosaved]' with you.pdf", 'Untitled document (2).gdoc', 'Untitled document (1).gdoc', 'Amazon_Complaints-20260414T034618Z-3-001.zip', 'Amazon_Complaints', 'wmn4&5.gdoc', '27 04.gdoc', 'IMG-20260505-WA0007.jpg', '5G_Full_Report.gdoc', 'Untitled document.gdoc', 'WMN assignme

In [ ]:
print(os.listdir('/content/drive/MyDrive/infosys_internship'))

['stations.csv', 'station_hour.csv', 'filter_city_data.py', 'delhi_station_hour.csv', 'add_coordinates.py', 'delhi_station_hour_with_coords.csv', 'add_weather.py', 'delhi_station_hour_with_weather.csv', 'clean_data.py', 'delhi_cleaned.csv', 'feature_engineering.py', 'delhi_features.csv', 'scale_and_split.py', 'scaler.pkl', 'delhi_train.csv', 'delhi_test.csv']


In [ ]:
"""
Trains an LSTM model to predict AQI 1 hour ahead, using delhi_train.csv /
delhi_test.csv from Google Drive. Fixed to avoid the "crashed after using
all available RAM" error - instead of building one giant array of all
sequences upfront (which needs 7+ GB just for X_train), this streams
sequences in small batches using tf.data, so memory stays low throughout.
"""

import pandas as pd
import pickle
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Input

# Update this if your Drive folder path is different
BASE_PATH = "/content/drive/MyDrive/infosys_internship"
SEQ_LEN = 24       # use the past 24 hours to predict the next hour's AQI
BATCH_SIZE = 64

train_df = pd.read_csv(f"{BASE_PATH}/delhi_train.csv", low_memory=False)
test_df = pd.read_csv(f"{BASE_PATH}/delhi_test.csv", low_memory=False)

exclude_cols = ["StationId", "StationName", "City", "State", "Datetime", "AQI_Bucket"]
feature_cols = [c for c in train_df.columns if c not in exclude_cols]
target_idx = feature_cols.index("AQI")
n_features = len(feature_cols)

def sequence_generator(df, feature_cols, target_idx, seq_len):
    """Yields ONE sequence at a time (not the whole dataset at once),
    per station so history never crosses between stations."""
    for station_id, group in df.groupby("StationId"):
        group = group.sort_values("Datetime")
        data = group[feature_cols].values.astype("float32")
        for i in range(len(data) - seq_len):
            yield data[i:i + seq_len], data[i + seq_len, target_idx]

output_signature = (
    tf.TensorSpec(shape=(SEQ_LEN, n_features), dtype=tf.float32),
    tf.TensorSpec(shape=(), dtype=tf.float32),
)

# tf.data streams + batches sequences on demand - RAM stays low no matter
# how many total sequences there are
train_ds = tf.data.Dataset.from_generator(
    lambda: sequence_generator(train_df, feature_cols, target_idx, SEQ_LEN),
    output_signature=output_signature,
).batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)

test_ds = tf.data.Dataset.from_generator(
    lambda: sequence_generator(test_df, feature_cols, target_idx, SEQ_LEN),
    output_signature=output_signature,
).batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)

# --- Build the LSTM model ---
model = Sequential([
    Input(shape=(SEQ_LEN, n_features)),
    LSTM(64, return_sequences=True),
    LSTM(32),
    Dense(16, activation="relu"),
    Dense(1),  # predicts scaled AQI
])
model.compile(optimizer="adam", loss="mse", metrics=["mae"])
model.summary()

# --- Train ---
history = model.fit(
    train_ds,
    validation_data=test_ds,
    epochs=15,
)
# --- Evaluate ---
test_loss, test_mae = model.evaluate(test_ds, verbose=0)
print(f"\nTest MSE (scaled): {test_loss:.4f}")
print(f"Test MAE (scaled): {test_mae:.4f}")

# --- Convert MAE back to real AQI units ---
with open(f"{BASE_PATH}/scaler.pkl", "rb") as f:
    scaler = pickle.load(f)

aqi_min = scaler.data_min_[target_idx]
aqi_max = scaler.data_max_[target_idx]
mae_real_units = test_mae * (aqi_max - aqi_min)
print(f"Test MAE (real AQI units): ~{mae_real_units:.1f} AQI points")

model.save("aqi_lstm_model.keras")
print("\nModel saved as aqi_lstm_model.keras")

from google.colab import files
files.download("aqi_lstm_model.keras")

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ lstm (LSTM)                     │ (None, 24, 64)         │        27,392 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_1 (LSTM)                   │ (None, 32)             │        12,416 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 16)             │           528 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 1)              │            17 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 40,353 (157.63 KB)

 Trainable params: 40,353 (157.63 KB)

 Non-trainable params: 0 (0.00 B)

...

/usr/local/lib/python3.13/dist-packages/keras/src/trainers/epoch_iterator.py:164: UserWarning: Your input ran out of data; interrupting training. Make sure that your dataset or generator can generate at least `steps_per_epoch * epochs` batches. You may need to use the `.repeat()` function when building your dataset.
  self._interrupted_warning()
/usr/local/lib/python3.13/dist-packages/keras/src/trainers/epoch_iterator.py:164: UserWarning: Your input ran out of data; interrupting training. Make sure that your dataset or generator can generate at least `steps_per_epoch * epochs` batches. You may need to use the `.repeat()` function when building your dataset.
  self._interrupted_warning()



Test MSE (scaled): nan
Test MAE (scaled): nan
Test MAE (real AQI units): ~nan AQI points

Model saved as aqi_lstm_model.keras


/usr/local/lib/python3.13/dist-packages/sklearn/base.py:380: InconsistentVersionWarning: Trying to unpickle estimator MinMaxScaler from version 1.9.0 when using version 1.6.1. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
from google.colab import files; files.download('aqi_lstm_model.keras')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
"""
Trains an LSTM model to predict AQI 1 hour ahead.

Fixes two issues from the previous run:
  1. NaN loss - caused by leftover missing values (some stations never
     measured certain pollutants like NH3, so interpolation couldn't
     fully fill them). Fixed by filling any remaining gaps with the
     TRAINING set's column averages (applied to both train and test,
     so no test-set info leaks into training).
  2. Slow training (~13 min/epoch) - caused by a pure-Python generator.
     Replaced with TensorFlow's built-in windowing, which is ~4x faster.
"""

import pandas as pd
import pickle
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Input

BASE_PATH = "/content/drive/MyDrive/infosys_internship"
SEQ_LEN = 24
BATCH_SIZE = 64

train_df = pd.read_csv(f"{BASE_PATH}/delhi_train.csv", low_memory=False)
test_df = pd.read_csv(f"{BASE_PATH}/delhi_test.csv", low_memory=False)

exclude_cols = ["StationId", "StationName", "City", "State", "Datetime", "AQI_Bucket"]
feature_cols = [c for c in train_df.columns if c not in exclude_cols]
target_idx = feature_cols.index("AQI")
n_features = len(feature_cols)

# --- Fill any remaining NaNs (train-only averages, applied to both) ---
fill_values = train_df[feature_cols].mean()
train_df[feature_cols] = train_df[feature_cols].fillna(fill_values)
test_df[feature_cols] = test_df[feature_cols].fillna(fill_values)
print("Remaining NaNs after fill - train:", train_df[feature_cols].isna().sum().sum(),
      "test:", test_df[feature_cols].isna().sum().sum())

# --- Build sequences efficiently, per station, using tf's built-in windowing ---
def build_dataset(df, feature_cols, target_idx, seq_len):
    datasets = []
    total = 0
    for station_id, group in df.groupby("StationId"):
        group = group.sort_values("Datetime")
        data = group[feature_cols].values.astype("float32")
        if len(data) <= seq_len:
            continue
        targets = data[seq_len:, target_idx]
        ds = tf.keras.utils.timeseries_dataset_from_array(
            data=data[:-1], targets=targets, sequence_length=seq_len, batch_size=None
        )
        datasets.append(ds)
        total += len(data) - seq_len
    combined = datasets[0]
    for d in datasets[1:]:
        combined = combined.concatenate(d)
    return combined, total

print("Building training sequences...")
train_ds, n_train_seq = build_dataset(train_df, feature_cols, target_idx, SEQ_LEN)
train_ds = train_ds.batch(BATCH_SIZE, drop_remainder=True).prefetch(tf.data.AUTOTUNE)

print("Building test sequences...")
test_ds, n_test_seq = build_dataset(test_df, feature_cols, target_idx, SEQ_LEN)
test_ds = test_ds.batch(BATCH_SIZE, drop_remainder=True).prefetch(tf.data.AUTOTUNE)

print(f"Train sequences: {n_train_seq} ({n_train_seq // BATCH_SIZE} steps/epoch)")
print(f"Test sequences: {n_test_seq} ({n_test_seq // BATCH_SIZE} steps)")

# --- Build the LSTM model ---
model = Sequential([
    Input(shape=(SEQ_LEN, n_features)),
    LSTM(64, return_sequences=True),
    LSTM(32),
    Dense(16, activation="relu"),
    Dense(1),
])
# clipnorm adds extra protection against exploding gradients going forward
model.compile(optimizer=tf.keras.optimizers.Adam(clipnorm=1.0), loss="mse", metrics=["mae"])
model.summary()

# --- Train ---
history = model.fit(train_ds, validation_data=test_ds, epochs=15)

# --- Evaluate ---
test_loss, test_mae = model.evaluate(test_ds, verbose=0)
print(f"\nTest MSE (scaled): {test_loss:.4f}")
print(f"Test MAE (scaled): {test_mae:.4f}")

with open(f"{BASE_PATH}/scaler.pkl", "rb") as f:
    scaler = pickle.load(f)
aqi_min = scaler.data_min_[target_idx]
aqi_max = scaler.data_max_[target_idx]
mae_real_units = test_mae * (aqi_max - aqi_min)
print(f"Test MAE (real AQI units): ~{mae_real_units:.1f} AQI points")

model.save("aqi_lstm_model.keras")
print("\nModel saved as aqi_lstm_model.keras")

from google.colab import files
files.download("aqi_lstm_model.keras")

Remaining NaNs after fill - train: 0 test: 0
Building training sequences...
Building test sequences...
Train sequences: 868962 (13577 steps/epoch)
Test sequences: 216602 (3384 steps)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ lstm (LSTM)                     │ (None, 24, 64)         │        27,392 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_1 (LSTM)                   │ (None, 32)             │        12,416 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 16)             │           528 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 1)              │            17 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 40,353 (157.63 KB)

 Trainable params: 40,353 (157.63 KB)

...

Epoch 1/15
13577/13577 ━━━━━━━━━━━━━━━━━━━━ 622s 45ms/step - loss: 0.0011 - mae: 0.0177 - val_loss: 4.2473e-04 - val_mae: 0.0143
Epoch 2/15
13577/13577 ━━━━━━━━━━━━━━━━━━━━ 605s 45ms/step - loss: 3.5025e-04 - mae: 0.0092 - val_loss: 4.8606e-04 - val_mae: 0.0156
Epoch 3/15
13577/13577 ━━━━━━━━━━━━━━━━━━━━ 604s 45ms/step - loss: 3.1734e-04 - mae: 0.0083 - val_loss: 5.2927e-04 - val_mae: 0.0165
Epoch 4/15
13577/13577 ━━━━━━━━━━━━━━━━━━━━ 607s 45ms/step - loss: 3.0240e-04 - mae: 0.0080 - val_loss: 5.1623e-04 - val_mae: 0.0163
Epoch 5/15
13577/13577 ━━━━━━━━━━━━━━━━━━━━ 600s 44ms/step - loss: 2.8436e-04 - mae: 0.0077 - val_loss: 5.2965e-04 - val_mae: 0.0167
Epoch 6/15
13577/13577 ━━━━━━━━━━━━━━━━━━━━ 601s 44ms/step - loss: 2.7275e-04 - mae: 0.0075 - val_loss: 5.1962e-04 - val_mae: 0.0166
Epoch 7/15
13577/13577 ━━━━━━━━━━━━━━━━━━━━ 602s 44ms/step - loss: 2.6029e-04 - mae: 0.0073 - val_loss: 4.6937e-04 - val_mae: 0.0157
Epoch 8/15
13577/13577 ━━━━━━━━━━━━━━━━━━━━ 598s 44ms/step - loss: 2.5094

/usr/local/lib/python3.13/dist-packages/sklearn/base.py:380: InconsistentVersionWarning: Trying to unpickle estimator MinMaxScaler from version 1.9.0 when using version 1.6.1. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(



Model saved as aqi_lstm_model.keras


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
"""
Loads your already-trained model and computes the full evaluation
metrics (RMSE, R2, MAPE, Accuracy %) WITHOUT retraining - since you
already have aqi_lstm_model.keras saved from your last run.
"""

import pandas as pd
import numpy as np
import pickle
import tensorflow as tf
from sklearn.metrics import r2_score, mean_absolute_percentage_error

BASE_PATH = "/content/drive/MyDrive/infosys_internship"
SEQ_LEN = 24
BATCH_SIZE = 64

model = tf.keras.models.load_model("aqi_lstm_model.keras")

test_df = pd.read_csv(f"{BASE_PATH}/delhi_test.csv", low_memory=False)
train_df = pd.read_csv(f"{BASE_PATH}/delhi_train.csv", low_memory=False)  # only needed to get the same fill values used before

exclude_cols = ["StationId", "StationName", "City", "State", "Datetime", "AQI_Bucket"]
feature_cols = [c for c in test_df.columns if c not in exclude_cols]
target_idx = feature_cols.index("AQI")

fill_values = train_df[feature_cols].mean()
test_df[feature_cols] = test_df[feature_cols].fillna(fill_values)

def build_dataset(df, feature_cols, target_idx, seq_len):
    datasets = []
    for station_id, group in df.groupby("StationId"):
        group = group.sort_values("Datetime")
        data = group[feature_cols].values.astype("float32")
        if len(data) <= seq_len:
            continue
        targets = data[seq_len:, target_idx]
        ds = tf.keras.utils.timeseries_dataset_from_array(
            data=data[:-1], targets=targets, sequence_length=seq_len, batch_size=None
        )
        datasets.append(ds)
    combined = datasets[0]
    for d in datasets[1:]:
        combined = combined.concatenate(d)
    return combined

test_ds = build_dataset(test_df, feature_cols, target_idx, SEQ_LEN).batch(BATCH_SIZE, drop_remainder=True).prefetch(tf.data.AUTOTUNE)

with open(f"{BASE_PATH}/scaler.pkl", "rb") as f:
    scaler = pickle.load(f)
aqi_min = scaler.data_min_[target_idx]
aqi_max = scaler.data_max_[target_idx]

y_true, y_pred = [], []
for xb, yb in test_ds:
    preds = model.predict(xb, verbose=0)
    y_true.extend(yb.numpy())
    y_pred.extend(preds.flatten())
y_true = np.array(y_true) * (aqi_max - aqi_min) + aqi_min
y_pred = np.array(y_pred) * (aqi_max - aqi_min) + aqi_min

mae_real = np.mean(np.abs(y_true - y_pred))
rmse_real = np.sqrt(np.mean((y_true - y_pred) ** 2))
r2 = r2_score(y_true, y_pred)
nonzero = y_true != 0
mape = mean_absolute_percentage_error(y_true[nonzero], y_pred[nonzero]) * 100
accuracy_pct = 100 - mape

print(f"MAE:  {mae_real:.1f} AQI points")
print(f"RMSE: {rmse_real:.1f} AQI points")
print(f"R2 score: {r2:.3f}")
print(f"MAPE: {mape:.1f}%")
print(f"Accuracy: {accuracy_pct:.1f}%")

/usr/local/lib/python3.13/dist-packages/sklearn/base.py:380: InconsistentVersionWarning: Trying to unpickle estimator MinMaxScaler from version 1.9.0 when using version 1.6.1. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(


MAE:  5.8 AQI points
RMSE: 8.2 AQI points
R2 score: 0.995
MAPE: 3.7%
Accuracy: 96.3%
